In [1]:
!pip install flask pyngrok scikit-learn pandas numpy joblib


In [3]:
import pandas as pd

# Load dataset
data = pd.read_csv('iris.csv')

# Display first rows
print(data.head())

# Check structure
print(data.info())


   sepallength  sepalwidth  petallength  petalwidth        class
0          5.1         3.5          1.4         0.2  Iris-setosa
1          4.9         3.0          1.4         0.2  Iris-setosa
2          4.7         3.2          1.3         0.2  Iris-setosa
3          4.6         3.1          1.5         0.2  Iris-setosa
4          5.0         3.6          1.4         0.2  Iris-setosa
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sepallength  150 non-null    float64
 1   sepalwidth   150 non-null    float64
 2   petallength  150 non-null    float64
 3   petalwidth   150 non-null    float64
 4   class        150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB
None


In [4]:
from sklearn.preprocessing import LabelEncoder

# Separate features and target
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

# Convert categorical labels to numeric
le = LabelEncoder()
y = le.fit_transform(y)

print("Classes:", le.classes_)


Classes: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier()
}

best_model = None
best_accuracy = 0

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    acc = accuracy_score(y_test, predictions)
    print(f"{name} Accuracy: {acc:.4f}")

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model

# Save best model
joblib.dump(best_model, "best_model.pkl")
joblib.dump(le, "label_encoder.pkl")

print("Best Model Saved Successfully!")


Logistic Regression Accuracy: 1.0000
SVM Accuracy: 1.0000
Random Forest Accuracy: 1.0000
Best Model Saved Successfully!


In [7]:
!pip install pyngrok


In [2]:
!ngrok config add-authtoken 39kFPFvL2fvH186JKDpsr5tQCuL_7sFbxYNwScpqcwrXrRrWw

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from flask import Flask, request, render_template_string
from pyngrok import ngrok
import numpy as np
import joblib

app = Flask(__name__)

model = joblib.load("best_model.pkl")
le = joblib.load("label_encoder.pkl")
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>Iris Flower Prediction</title>
    <style>
        body {
            margin: 0;
            font-family: Arial, Helvetica, sans-serif;
            background: linear-gradient(to right, #4e73df, #1cc88a);
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
        }

        .card {
            background: white;
            padding: 30px;
            width: 350px;
            border-radius: 15px;
            box-shadow: 0px 10px 25px rgba(0,0,0,0.2);
            text-align: center;
        }

        h2 {
            margin-bottom: 20px;
            color: #333;
        }

        input {
            width: 90%;
            padding: 10px;
            margin: 8px 0;
            border: 1px solid #ccc;
            border-radius: 8px;
            font-size: 14px;
        }

        input:focus {
            border-color: #4e73df;
            outline: none;
        }

        button {
            width: 95%;
            padding: 10px;
            margin-top: 15px;
            border: none;
            border-radius: 8px;
            background-color: #4e73df;
            color: white;
            font-size: 15px;
            cursor: pointer;
            transition: 0.3s;
        }

        button:hover {
            background-color: #2e59d9;
        }

        .result {
            margin-top: 20px;
            font-size: 18px;
            font-weight: bold;
            color: #1cc88a;
        }

        .footer {
            margin-top: 15px;
            font-size: 12px;
            color: #777;
        }
    </style>
</head>

<body>
    <div class="card">
        <h2>🌸 Iris Flower Prediction</h2>
        <form method="POST">
            <input type="text" name="sl" placeholder="Sepal Length (cm)" required>
            <input type="text" name="sw" placeholder="Sepal Width (cm)" required>
            <input type="text" name="pl" placeholder="Petal Length (cm)" required>
            <input type="text" name="pw" placeholder="Petal Width (cm)" required>
            <button type="submit">Predict</button>
        </form>

        {% if prediction %}
            <div class="result">
                {{ prediction }}
            </div>
        {% endif %}

        <div class="footer">
            Machine Learning Web App using Flask
        </div>
    </div>
</body>
</html>
"""


@app.route("/", methods=["GET", "POST"])
def home():
    prediction = ""
    if request.method == "POST":
        sl = float(request.form["sl"])
        sw = float(request.form["sw"])
        pl = float(request.form["pl"])
        pw = float(request.form["pw"])

        input_data = np.array([[sl, sw, pl, pw]])
        pred = model.predict(input_data)
        prediction = "Predicted Species: " + le.inverse_transform(pred)[0]

    return render_template_string(html_template, prediction=prediction)

# Kill old tunnels if any
ngrok.kill()

# Create new tunnel
public_url = ngrok.connect(5000)
print("🌍 OPEN THIS LINK:", public_url)

app.run(port=5000)


🌍 OPEN THIS LINK: NgrokTunnel: "https://uncaustically-cyclonical-maliyah.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [16/Feb/2026 09:18:24] "GET / HTTP/1.1" 200 -
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [16/Feb/2026 09:18:35] "POST / HTTP/1.1" 200 -
